# s3:// vs https:// virtual references: read test

Part of [ocean-icechunks-skills#27](https://github.com/ocean-icechunks/ocean-icechunks-skills/issues/27).

Two pairs of small virtual Icechunk stores, each pair pointing at **the same source files** and differing
only in how the references are written (**H** = `https://`, Icechunk HTTP store; **S** = `s3://`, Icechunk S3 store, anonymous):

| dataset | sources | H prefix | S prefix | chunks |
|---|---|---|---|---|
| `roms` | 8 PacIOOS ROMS reanalysis files on Source Cooperative | `https://data.source.coop/ocean-icechunks/pacioos/` | `s3://ocean-icechunks/pacioos/` (endpoint `data.source.coop`) | 64 × ~8.3 MB, uncompressed |
| `oisst` | 64 days of NOAA OISST v2.1 on NODD (AWS us-east-1) | `https://noaa-cdr-sea-surface-temp-optimum-interpolation-pds.s3.amazonaws.com/` | `s3://noaa-cdr-sea-surface-temp-optimum-interpolation-pds/` | 64 × ~1 MB, gzip |

The notebook reads both pairs **with no credentials** and times the same reads, alternating H and S each round.
`roms` pulls about 530 MB per bulk read, so a slow connection will take a while.
Run top to bottom (Colab: *Runtime → Run all*), then paste the last cell's output into the issue.

In [ ]:
%pip install -q icechunk==2.3.1 zarr==3.4.2 xarray==2026.9.0

In [ ]:
import json, platform, socket, statistics, time, warnings
warnings.simplefilter("ignore")
import icechunk, numpy as np, xarray as xr, zarr
print("icechunk", icechunk.__version__, "zarr", zarr.__version__, "xarray", xr.__version__)

In [ ]:
ROOT = "https://data.source.coop/ocean-icechunks/test-repo/s3-vs-https"
# name: (store folder, variable to read)
DATASETS = {
    "roms": ("roms_reanalysis_8day", "temp"),     # Source Coop sources, ~8.3 MB uncompressed chunks
    "oisst": ("oisst_nodd_64day", "sst"),         # NODD AWS us-east-1 sources, ~1 MB gzip chunks
}
DATASET = "roms"


def authorize(repo):
    """Anonymous access for every container the store names, by its type."""
    auth = {}
    for prefix, c in (repo.config.virtual_chunk_containers or {}).items():
        if prefix.startswith("s3://"):
            auth[prefix] = icechunk.credentials.s3_anonymous_credentials()
        elif prefix.startswith(("http://", "https://")):
            auth[prefix] = icechunk.credentials.HttpAccess
        else:
            raise ValueError(f"no anonymous rule for {prefix}")
    return repo.reopen(authorize_virtual_chunk_access=icechunk.containers_credentials(auth))


def open_ds(kind, dataset=None):
    folder = DATASETS[dataset or DATASET][0]
    repo = authorize(icechunk.Repository.open(icechunk.http_storage(f"{ROOT}/{folder}/{kind}")))
    return xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks=None)


def pick_var(ds, prefer="temp"):
    """`prefer` if present, else the largest variable with time (ties broken by name),
    so both stores always read the same variable."""
    if prefer in ds.data_vars:
        return prefer
    cands = sorted(v for v in ds.data_vars if "time" in ds[v].dims)
    return max(cands, key=lambda v: (ds[v].ndim, ds[v].size))


def cases(ds, var):
    a = ds[var]
    mid = {d: a.sizes[d] // 2 for d in a.dims if d != "time"}
    surf = {d: 0 for d in ("depth", "zlev") if d in a.dims}
    return {
        "one_slice": lambda: a.isel(time=0, **surf).values,
        "point_series": lambda: a.isel(**mid).values,
        "surface_all_times": lambda: a.isel(**surf).values,
        "full_variable": lambda: a.values,
    }


def timed(fn):
    t0 = time.perf_counter()
    out = fn()
    return time.perf_counter() - t0, out


def run(repeats=3, label=None, kinds=("H", "S"), dataset=None):
    dataset = dataset or DATASET
    zarr.config.set({"async.concurrency": 64})
    results = {"label": label or socket.gethostname(), "dataset": dataset,
               "python": platform.python_version(),
               "icechunk": icechunk.__version__, "repeats": repeats, "times": {}}
    checks = {}
    # Alternate H and S each round so slow network moments hit both alike.
    for r in range(repeats):
        for kind in kinds:
            dt, ds = timed(lambda: open_ds(kind, dataset))
            results["times"].setdefault(kind, {}).setdefault("open", []).append(dt)
            var = pick_var(ds, DATASETS[dataset][1])
            for name, fn in cases(ds, var).items():
                dt, arr = timed(fn)
                results["times"][kind].setdefault(name, []).append(dt)
                if r == 0:
                    checks.setdefault(name, {})[kind] = (float(np.nansum(arr)), arr.shape)
    results["variable"] = var
    results["same_values"] = all(len({str(v) for v in c.values()}) == 1 for c in checks.values())
    return results


def report(results):
    print(f"{results['label']} [{results.get('dataset', 'roms')}]: icechunk {results['icechunk']}, variable {results['variable']}, "
          f"{results['repeats']} repeats; H and S values identical: {results['same_values']}")
    t = results["times"]
    print(f"{'read':20s} {'H https (s)':>22s} {'S s3 (s)':>22s} {'S/H':>6s}")
    for name in t["H"]:
        h, s = t["H"][name], t["S"][name]
        fmt = lambda x: f"{statistics.median(x):6.2f} [{min(x):.2f}-{max(x):.2f}]"
        print(f"{name:20s} {fmt(h):>22s} {fmt(s):>22s} "
              f"{statistics.median(s) / statistics.median(h):6.2f}")
    print("median [min-max]; first round is cold, later rounds may be warmer at the server/CDN")

## Where is this running?
Set `LABEL` to say where, e.g. `colab`, `laptop-home-wifi`, `laptop-office`.

In [ ]:
LABEL = "colab"
print(LABEL)

## Can both stores be read without credentials? Same values?

In [ ]:
# Both pairs open without credentials and give the same value at one ocean point.
for dataset, (folder, var) in DATASETS.items():
    for kind in ("H", "S"):
        ds = open_ds(kind, dataset)
        a = ds[var].isel(time=0)
        a = a.isel({d: 0 for d in ("depth", "zlev") if d in a.dims})
        print(dataset, kind, dict(ds.sizes), f"{var} at centre:", float(a.isel(lat=a.sizes["lat"] // 2, lon=a.sizes["lon"] // 2)))

## Timing (5 rounds)

In [ ]:
all_results = {}
for dataset in DATASETS:
    all_results[dataset] = run(repeats=5, label=LABEL, dataset=dataset)
    report(all_results[dataset])
    print()

## Does Zarr concurrency matter? (whole variable, fresh open each read, best of 2)

In [ ]:
for dataset, (folder, var) in DATASETS.items():
    conc_rows = {}
    for conc in (8, 16, 64, 128):
        zarr.config.set({"async.concurrency": conc})
        row = {}
        for kind in ("H", "S"):
            ts = []
            for _ in range(2):
                ds = open_ds(kind, dataset)   # fresh open: xarray caches .values on a DataArray
                t = time.perf_counter(); ds[var].values; ts.append(time.perf_counter() - t)
            row[kind] = round(min(ts), 2)
        conc_rows[conc] = row
        print(dataset, "concurrency", conc, row)
    all_results[dataset]["concurrency"] = conc_rows

## Paste this into the issue

In [ ]:
print(json.dumps(all_results, indent=1))